# 02 — Hierarchical Moderated Regression, Simple Slopes & Johnson–Neyman

**Model:** ODP = *b₀* + *b₁*(ALV) + *b₂*(AI Agency) + *b₃*(ALV × AI Agency) + *e*

## Purpose
This notebook implements the full moderation analysis pipeline for an APA-style manuscript:

1. **Hierarchical moderated regression**
   - *Model 1:* main effect of ALV (centered)
   - *Model 2:* adds main effect of AI Agency (centered)
   - *Model 3:* adds the ALV × AI Agency interaction term
2. **Simple slopes analysis** at the mean and ±1 *SD* of the moderator, with significance tests.
3. **Johnson–Neyman technique** — the range of the moderator for which the simple slope is significant.
4. **Path model plot** of the moderation model.

> **APA note:** Center predictors before computing the interaction term to reduce nonessential collinearity; report *R²* change (Δ*R²*), *F*-change, and unstandardized *b* with *SE* and 95% CI for the interaction.

In [ ]:
# ============================================================
# Setup
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
RNG = np.random.default_rng = "/mnt/data/raw_data_template (1).csv"
print("Environment ready.")print("Environment ready.")

In [ ]:
# ============================================================
# 1. Load data (same preparation as Notebook 01)
# ============================================================
df = pd.read_csv(DATA_PATH)
N_VARS = ["ALV", "AI_Agency", "ODP"]

if df[N_VARS].isna().all().all():
    # Demonstration data consistent with the published correlations
    N = 60
    z = RNG.multivariate_normal([0,0,0], np.array([
        [1,-.28,-.54],[-.28,1,.46],[-.54,.46,1]]), size=N)
    def to_scale(x, m, sd, lo, hi):
        return np.clip(m + sd*stats.zscore(x), lo, hi)
    df["ALV"] = to_scale(z[:,0], 3.0, 0.7, 1, 5)
    df["AI_Agency"] = to_scale(z[:,1], 4.1, 0.6, 1, 5)
    df["ODP"] = to_scale(z[:,2], 2.95, 0.75, 1, 5)
    print("NOTE: demonstration data generated (blank template). N =", N)
else:
    df = df.dropna(subset=N_VARS).reset_index(drop=True)
    print(f"Observed data. N = {len(df)}")

# Mean-center predictors before forming the interaction (APA recommendation)
df["ALV_c"] = df["ALV"] - df["ALV"].mean()
df["AI_c"] = df["AI_Agency"] - df["AI_Agency"].mean()
df["Interaction"] = df["ALV_c"] * df["AI_c"]
df.head()

In [ ]:
# ============================================================
# 2. Hierarchical moderated regression (Models 1, 2, 3)
# ============================================================
models = {
    "Model 1": smf.ols("ODP ~ ALV_c", data=df).fit(),
    "Model 2": smf.ols("ODP ~ ALV_c + AI_c", data=df).fit(),
    "Model 3": smf.ols("ODP ~ ALV_c * AI_c", data=df).fit(),
}

rows, prev_r2 = [], 0.0
for name, m in models.items():
    r2, r2adj = m.rsquared, m.rsquared_adj
    dr2 = r2 - prev_r2
    df1 = int(m.df_model) - int(models["Model 1"].df_model if name != "Model 1" else 0)
    # F-change test
    if name == "Model 1":
        Fc, pF = m.fvalue, m.f_pvalue
    else:
        # nested model comparison
        prev = models["Model 1" if name == "Model 2" else "Model 2"]
        from statsmodels.stats.anova import anova_lm
        aov = anova_lm(prev, m)
        Fc, pF = aov["F"].iloc[1], aov["Pr(>F)"].iloc[1]
    prev_r2 = r2
    for term in m.params.index:
        rows.append({
            "Model": name, "Predictor": term,
            "B": m.params[term], "SE": m.bse[term],
            "Beta": m.params.get(term, np.nan),
            "t": m.tvalues[term], "p": m.pvalues[term],
            "CI_low": m.conf_int().loc[term, 0], "CI_high": m.conf_int().loc[term, 1],
        })
    print(f"{name}: R² = {r2:.3f}, adj. R² = {r2adj:.3f}, ΔR² = {dr2:.3f}, "
          f"F-change = {Fc:.2f}, p = {pF:.4f}")

reg_table = pd.DataFrame(rows)
display(reg_table.round(3))

print("\nAPA-style narrative check: moderation is supported if the interaction term "
      "(ALV_c:AI_c in Model 3) is significant (p < .05).")

In [ ]:
# ============================================================
# 3. Simple slopes analysis (-1 SD, Mean, +1 SD of moderator)
# ============================================================
mod_sd = df["AI_c"].std(ddof=1)
levels = {"Low (-1 SD)": -mod_sd, "Mean": 0.0, "High (+1 SD)": mod_sd}

print(f"Moderator (AI Agency, centered) SD = {mod_sd:.3f}\n")
slope_rows = []
for label, w in levels.items():
    df[f"W_{label}"] = df["AI_c"] - w          # re-center moderator at level
    m = smf.ols(f"ODP ~ ALV_c * W_{label}", data=df).fit()
    b, se, t, p = (m.params[f"ALV_c"], m.bse[f"ALV_c"],
                   m.tvalues[f"ALV_c"], m.pvalues[f"ALV_c"])
    ci = m.conf_int().loc[f"ALV_c"].tolist()
    slope_rows.append({"Moderator_Level": label, "AI_Agency_value": w + df["AI_Agency"].mean(),
                       "Slope": b, "SE": se, "t": t, "p": p,
                       "CI_low": ci[0], "CI_high": ci[1]})
    print(f"{label:<12} slope = {b:.3f}, SE = {se:.3f}, t({m.df_resid:.0f}) = {t:.2f}, "
          f"p = {p:.4f}, 95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")

slopes = pd.DataFrame(slope_rows)
display(slopes.round(3))

In [ ]:
# ============================================================
# 4. Johnson-Neyman technique
# ============================================================
# Find values of the moderator where the simple slope of ALV is
# significant (p = .05) — i.e., where b1 + b3*W / SE(W) exceeds
# the critical t. Closed-form solution following Bauer & Curran (2005).
m3 = models["Model 3"]
b1, b2, b3 = m3.params["ALV_c"], m3.params["AI_c"], m3.params["ALV_c:AI_c"]
cov = m3.cov_params()
var_b1, var_b3, cov_b1b3 = cov.loc["ALV_c","ALV_c"], cov.loc["ALV_c:AI_c","ALV_c:AI_c"], cov.loc["ALV_c","ALV_c:AI_c"]
tcrit = stats.t.ppf(1 - .05/2, int(m3.df_resid))

a = var_b3
b = 2 * cov_b1b3
c = var_b1 - tcrit**2 * (var_b1 - 2*cov_b1b3*0 + 0)  # corrected below
c = var_b1 - (tcrit**2) * 0 - 0
# Proper quadratic: Var(b1 + b3*W) = tcrit^2 * SE^2 boundary condition:
# (b1 + b3*W)^2 / (var_b1 + 2*cov*W + var_b3*W^2) = tcrit^2
# -> (b3^2 - tcrit^2*var_b3) W^2 + 2(b1*b3 - tcrit^2*cov) W + (b1^2 - tcrit^2*var_b1) = 0
A = b3**2 - tcrit**2 * var_b3
B = 2 * (b1*b3 - tcrit**2 * cov_b1b3)
C = b1**2 - tcrit**2 * var_b1
disc = B**2 - 4*A*C
print(f"b1 = {b1:.3f}, b3 = {b3:.3f}, t-crit = {tcrit:.3f}")
if disc >= 0 and A != 0:
    W1, W2 = ((-B - np.sqrt(disc)) / (2*A)), ((-B + np.sqrt(disc)) / (2*A))
    W1, W2 = sorted([W1, W2])
    lo_raw, hi_raw = W1 + df["AI_Agency"].mean(), W2 + df["AI_Agency"].mean()
    print(f"Johnson-Neyman transition points (centered W): {W1:.3f}, {W2:.3f}")
    print(f"  -> on raw AI Agency scale: {lo_raw:.3f} and {hi_raw:.3f}")
    W_grid = np.linspace(df["AI_c"].min(), df["AI_c"].max(), 400)
    slope = b1 + b3*W_grid
    varW = var_b1 + 2*cov_b1b3*W_grid + var_b3*W_grid**2
    seW = np.sqrt(varW)
    sig = np.abs(slope/seW) > tcrit
    plt.figure(figsize=(8,5))
    plt.plot(W_grid + df["AI_Agency"].mean(), slope, lw=2, label="Simple slope of ALV")
    plt.fill_between(W_grid + df["AI_Agency"].mean(), slope,
                     where=sig, alpha=.18, color="steelblue", label="Region of significance (p < .05)")
    plt.axhline(0, color="grey", lw=.8)
    for x in (lo_raw, hi_raw):
        plt.axvline(x, color="crimson", ls="--", lw=1)
    plt.xlabel("AI Agency (moderator, raw scale)")
    plt.ylabel("Simple slope of ALV on ODP")
    plt.title("Johnson-Neyman Region of Significance", fontweight="bold")
    plt.legend()
    plt.tight_layout()
    plt.savefig("/mnt/data/johnson_neyman.png", bbox_inches="tight")
    plt.show()
else:
    print("No real roots: the simple slope is significant at all (or no) values of the moderator.")

In [ ]:
# ============================================================
# 5. Interaction plot (simple slopes visualization)
# ============================================================
plt.figure(figsize=(7.5, 5.5))
xg = np.linspace(df["ALV"].min(), df["ALV"].max(), 100)
colors = {"Low (-1 SD)": "#1f77b4", "Mean": "#555555", "High (+1 SD)": "#d62728"}
m3 = models["Model 3"]
for label, w in levels.items():
    y = m3.params["Intercept"] + m3.params["ALV_c"]*(xg - df["ALV"].mean()) + \
        m3.params["AI_c"]*w + m3.params["ALV_c:AI_c"]*(xg - df["ALV"].mean())*w
    plt.plot(xg, y, lw=2.2, color=colors[label],
             label=f"{label} (AI Agency)")
plt.xlabel("Artificial Linguistic Veneer (ALV)")
plt.ylabel("Oral Defense Performance (ODP)")
plt.title("ALV × AI Agency Interaction: Simple Slopes", fontweight="bold")
plt.legend(title="Moderator level")
plt.tight_layout()
plt.savefig("/mnt/data/simple_slopes_plot.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 6. Path model plot of the moderation model
# ============================================================
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")

def box(x, y, text, color="#eef3fb"):
    ax.add_patch(mpatches.FancyBboxPatch((x-1.15, y-.5), 2.3, 1.0,
                 boxstyle="round,pad=0.08", fc=color, ec="#2b4a6f", lw=1.4))
    ax.text(x, y, text, ha="center", va="center", fontsize=10, fontweight="bold")

box(1.8, 4.6, "ALV\n(X)")
box(1.8, 1.4, "AI Agency\n(W)")
box(7.8, 3.0, "Oral Defense\nPerformance (Y)")

ax.annotate("", xy=(6.6, 3.2), xytext=(2.9, 4.4),
            arrowprops=dict(arrowstyle="-|>", lw=1.8, color="#2b4a6f"))
ax.text(4.4, 4.3, f"b1 = {b1:.2f}", fontsize=10, color="#2b4a6f")
ax.annotate("", xy=(6.6, 2.8), xytext=(2.9, 1.6),
            arrowprops=dict(arrowstyle="-|>", lw=1.8, color="#2b4a6f"))
ax.text(4.4, 1.8, f"b2 = {b2:.2f}", fontsize=10, color="#2b4a6f")
# Interaction arrow
ax.add_patch(mpatches.FancyBboxPatch((3.6, 3.0), 1.2, 0.7,
             boxstyle="round,pad=0.05", fc="#fff3cd", ec="#b8860b", lw=1.3))
ax.text(4.2, 3.35, f"X×W\nb3 = {b3:.2f}", ha="center", va="center", fontsize=9)
ax.annotate("", xy=(4.15, 3.4), xytext=(2.95, 4.3),
            arrowprops=dict(arrowstyle="-", lw=1.2, color="#b8860b"))
ax.annotate("", xy=(4.15, 3.2), xytext=(2.95, 1.7),
            arrowprops=dict(arrowstyle="-", lw=1.2, color="#b8860b"))
ax.annotate("", xy=(6.6, 3.0), xytext=(4.9, 3.35),
            arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#b8860b"))
p_int = m3.pvalues["ALV_c:AI_c"]
ax.text(5, 0.35, f"Interaction effect: b3 = {b3:.3f}, p = {p_int:.4f} "
        + ("→ significant moderation" if p_int < .05 else "→ no significant moderation"),
        ha="center", fontsize=10, style="italic")
ax.set_title("Statistical Moderation Path Model", fontweight="bold")
plt.tight_layout()
plt.savefig("/mnt/data/path_model.png", bbox_inches="tight")
plt.show()

print("\nNotebook 02 complete. Figures saved: johnson_neyman.png, simple_slopes_plot.png, path_model.png")